# 03 — Model Evaluation

Load a trained checkpoint and run evaluation:
- Per-class accuracy and macro F1-score  
- Confusion matrix  
- Classification report

In [ ]:
import sys
sys.path.insert(0, '../src')

from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score, accuracy_score
)
from torch.utils.data import DataLoader

from data import EMOTION_MAP, RavdessDataset, TorontoDataset
from train import CombinedDataModule, EmotionLitModel

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')

EMOTION_LABELS = [EMOTION_MAP[i + 1] for i in range(len(EMOTION_MAP))]

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────
CKPT_PATH   = Path('../logs/checkpoints/last.ckpt')   # change as needed
RAVDESS_DIR = '../data/ravdess'
TESS_DIR    = '../data/toronto'
BATCH_SIZE  = 32

In [ ]:
# ── Load model & datamodule ────────────────────────────────────────────────
if not CKPT_PATH.exists():
    print(f'Checkpoint not found at {CKPT_PATH}. Using random weights for demo.')
    from model import EmotionModel
    raw_model = EmotionModel(num_classes=8)
else:
    lit = EmotionLitModel.load_from_checkpoint(str(CKPT_PATH), map_location='cpu')
    raw_model = lit.model
    print(f'Loaded checkpoint: {CKPT_PATH}')

raw_model.eval()

datamodule = CombinedDataModule(
    ravdess_dir=RAVDESS_DIR, tess_dir=TESS_DIR,
    batch_size=BATCH_SIZE, num_workers=0,
)
datamodule.setup()

In [ ]:
# ── Run inference on test split ────────────────────────────────────────────
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in datamodule.test_dataloader():
        specs = batch.spectrogram
        log_probs = raw_model(specs)
        preds = log_probs.argmax(dim=-1)
        all_preds.extend(preds.tolist())
        all_labels.extend(batch.label.tolist())

print(f'Test samples: {len(all_labels)}')
print(f'Accuracy:     {accuracy_score(all_labels, all_preds):.4f}')
print(f'Macro F1:     {f1_score(all_labels, all_preds, average="macro"):.4f}')

In [ ]:
# ── Confusion matrix ───────────────────────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, data, fmt, title in [
    (axes[0], cm,      'd',    'Confusion Matrix (counts)'),
    (axes[1], cm_norm, '.2f',  'Confusion Matrix (normalised)'),
]:
    sns.heatmap(
        data, annot=True, fmt=fmt, ax=ax, cmap='Blues',
        xticklabels=EMOTION_LABELS, yticklabels=EMOTION_LABELS,
    )
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(title)

plt.tight_layout()
plt.show()

In [ ]:
# ── Full classification report ─────────────────────────────────────────────
print(classification_report(
    all_labels, all_preds,
    target_names=EMOTION_LABELS,
    digits=4,
))

In [ ]:
# ── Per-class F1 bar chart ─────────────────────────────────────────────────
per_class_f1 = f1_score(all_labels, all_preds, average=None)

colors = ['#EF5350' if f < 0.80 else '#66BB6A' for f in per_class_f1]
plt.figure(figsize=(10, 4))
bars = plt.bar(EMOTION_LABELS, per_class_f1, color=colors, edgecolor='none')
plt.axhline(0.80, color='gray', linestyle='--', label='Target (0.80)')
plt.ylabel('F1-Score')
plt.title('Per-Emotion F1-Score (macro target > 0.80)')
plt.legend()
plt.ylim(0, 1.05)
for bar, val in zip(bars, per_class_f1):
    plt.text(bar.get_x() + bar.get_width() / 2, val + 0.01, f'{val:.2f}',
             ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()